# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your product IDs and desired actions
3. Run cell 4 — prices are computed per product × cohort (using cohort-specific market data)
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

> Each product is automatically expanded to all 9 cohorts. Market-based actions use cohort-specific prices.  
> Main/general cohorts (695, 61, 699, 697, 698, 696) are auto-mirrored by the push handler.

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `step_up` | Next tier above current price in the price list |
| `step_down` | Next tier below current price in the price list |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

> Only SKUs with stock > 0 are pushed.

In [1]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


Push Prices Handler loaded at 2026-04-22 12:31:33 Cairo time
✓ API credentials loaded successfully
✓ Google Sheets client initialized
Ready. Today: 2026-04-22


In [2]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f'''

    SELECT 
        cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        AVG(cpu.price) AS current_price
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
        AND cpu.created_at::DATE <> '2023-07-31'
        AND cpu.is_customized = TRUE
        and basic_unit_count = 1 
    GROUP BY 1,2,3,4

'''
df_prices = query_snowflake(PRICES_QUERY)
#df_prices = setup_environment_2.dwh_pg_query(PRICES_QUERY, columns=['cohort_id','product_id','packing_unit_id','basic_unit_count','current_price'])
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = '''
WITH sales_check AS (
    SELECT DISTINCT
        pso.product_id,
        pso.packing_unit_id,
        SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= CURRENT_DATE - 60 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY 1, 2
)
SELECT product_id, packing_unit_id, basic_unit_count
FROM (
    SELECT *, MAX(nmv) OVER (PARTITION BY product_id, is_basic_unit) AS top_nmv
    FROM (
        SELECT 
            pup.product_id,
            pup.packing_unit_id,
            pup.basic_unit_count,
            pup.is_basic_unit,
            COUNT(DISTINCT CASE WHEN pup.basic_unit_count = 1 THEN pup.packing_unit_id END) 
                OVER (PARTITION BY pup.product_id) AS total_basic,
            COALESCE(sc.nmv, 0) AS nmv
        FROM packing_unit_products pup
        LEFT JOIN sales_check sc 
            ON sc.product_id = pup.product_id 
            AND sc.packing_unit_id = pup.packing_unit_id
        WHERE pup.deleted_at IS NULL
    )
)
WHERE nmv = top_nmv OR (top_nmv = 0 AND total_basic <= 1)
'''
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading stocks...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
        AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Queries Module | Timezone: America/Los_Angeles
✅ UTH and Last Hour functions defined
✅ Yesterday closing stock function defined
Fixed price & cart rule functions defined
get_commercial_min_prices() defined
get_commercial_price_ups() defined
get_margin_boundaries_region() defined
get_margin_boundaries_global() defined

QUERIES MODULE READY

Live Data Functions:
  • get_current_stocks()
  • get_packing_units()
  • get_current_prices()
  • get_current_wac()
  • get_current_cart_rules()

UTH Performance Functions:
  • get_uth_performance()         - UTH qty/retailers (Snowflake)
  • get_hourly_distribution()     - Historical hour contributions (Snowflake)
  • get_last_hour_performance()   - Last hour qty/retailers (DWH)

Stock Functions:
  • get_yesterday_closing_stock() - Yesterday closing stock with parent WH mapping

Override Functions:
  • get_fixed_prices()

In [3]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

# V2 market_data returns (product_id, region, price_tiers, wac_p, target_margin, num_sources, market_data_source)
# Expand to cohorts for lookup
df_market_cohorts = expand_to_cohorts(market_data)

# Lookup is per (product_id, cohort_id)
lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# Fill target_margin for rows without V2 data
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.05)
lookup.drop(columns=[c for c in lookup.columns if c.startswith('target_bm') or c.startswith('cat_target')], inplace=True, errors='ignore')

# Derive market percentile prices from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# Merge current base-unit price per (product_id, cohort_id)
base_prices = df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']].drop_duplicates()
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# Merge stocks (sum across warehouses per product for display; warehouse-level for push filtering)
wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
product_stocks = df_stocks.groupby('product_id')['stocks'].sum().reset_index().rename(columns={'stocks': 'total_stocks'})
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

Lookup table: 233420 rows (product x cohort)
  With market data: 44856
  With WAC: 76046
  With target margin: 233420


In [4]:
lookup[(lookup['total_stocks']>0)].to_excel('manual_data.xlsx')

In [ ]:
#lookup.to_excel('manual_data.xlsx')

In [ ]:
#lookup[(lookup['brand']=='حلو الشام')&(lookup['total_stocks']>0)]

In [5]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin',
#                  'step_up', 'step_down', or a number (fixed price)
#
# step_up: next tier above current price in the price_tiers list
# step_down: next tier below current price in the price_tiers list
# Each product is automatically expanded to ALL cohorts.
# Market-based actions use the cohort-specific market price.
# Only SKUs with stock > 0 are pushed.

PUSH_LIST = [
(8285,703,'market_min'),
(21269,701,'market_min'),
(21269,703,'market_min'),
(12675,701,'market_50'),
(2287,1124,'market_min'),
(5584,700,'market_min'),
(6569,700,'market_50'),
(6569,701,'market_50'),
(12870,701,'market_min'),
(10716,701,'market_min'),
(952,701,'market_min'),
(952,1123,'market_min'),
(1622,704,'market_min'),
(3896,1124,'market_25'),
(5208,704,'market_75'),
(3908,700,'market_75'),
(3908,701,'market_min'),
(5243,701,'market_min'),
(5243,702,'market_min'),
(5243,703,'market_25'),
(5243,704,'market_25'),
(5243,1123,'market_min'),
(5243,1124,'market_min'),
(11150,703,'market_min'),
(3910,701,'market_min'),
(7703,1123,'market_min'),
(12304,701,'market_min'),
(12304,1123,'market_min'),
(12658,702,'market_25'),
(12658,703,'market_min'),
(12658,1123,'market_25'),
(12658,1125,'market_min'),
(1024,701,'market_50'),
(10599,701,'market_min'),
(9830,701,'market_min'),
(9830,704,'market_min'),
(9995,1125,'market_min'),
(5976,1124,'market_min'),
(5976,1125,'market_min'),
(10397,704,'market_50'),
(10397,1126,'market_50'),
(12338,704,'market_25'),
(1124,701,'market_25'),
(1124,703,'market_min'),
(1124,704,'market_min'),
(9828,704,'market_min'),
(414,701,'market_min'),
(13028,701,'market_50'),
(13034,701,'market_min'),
(13034,702,'market_min'),
(247,703,'market_min'),
(8923,700,'market_min'),
(8923,701,'market_min'),
(8923,702,'market_min'),
(8923,703,'market_min'),
(8923,1126,'market_min'),
(97,700,'market_min'),
(97,701,'market_min'),
(97,703,'market_min'),
(995,701,'market_min'),
(995,703,'market_min'),
(995,704,'market_min'),
(998,701,'market_min'),
(74,704,'market_min'),
(1407,701,'market_50'),
(6792,701,'market_25'),
(3985,703,'market_min'),
(11677,701,'market_50'),
(10719,703,'market_min'),
(10148,700,'market_min'),
(10148,701,'market_min'),
(10148,703,'market_min'),
(13489,1124,'market_50'),
(11948,703,'market_min'),
(13024,702,'market_min'),
(13024,703,'market_25'),
(13024,704,'market_min'),
(13024,1124,'market_min'),
(13024,1125,'market_min'),
(10482,701,'market_min'),
(6560,700,'market_min'),
(6565,700,'market_min'),
(6800,700,'market_50'),
(3028,701,'market_50'),
(3028,704,'market_50'),
(3028,1125,'market_50'),
(3059,703,'market_min'),
(3495,702,'market_min'),
(1080,701,'market_50'),
(3980,701,'market_min'),
(3980,704,'market_min'),
(3980,1125,'market_min'),
(9910,700,'market_min'),
(9910,701,'market_min'),
(6627,701,'market_min'),
(5654,1123,'market_min'),
(6034,700,'market_min'),
(6034,701,'market_min'),
(8155,701,'market_min'),
(8155,702,'market_min'),
(8155,703,'market_min'),
(3557,1126,'market_min'),
(13243,702,73.1036990775501),
(59,1126,'market_min'),
(13263,704,'market_min'),
(205,701,'market_25'),
(3840,701,'market_25'),
(3840,703,'market_25'),
(10536,701,'market_min'),
(416,702,'market_min'),
(6072,701,'market_min'),
(1050,701,'market_50'),
(11936,701,'market_50'),
(11936,704,'market_25'),
(12496,703,'market_50'),
(12496,1125,'market_50'),
(903,1126,'market_min'),
(11489,701,'market_min'),
(1668,1125,'market_min'),
(12050,704,'market_min'),
(8635,701,'market_min'),
(9491,1125,'market_min'),
(2104,700,'market_min'),
(9085,701,'market_min'),
(7704,1123,'market_min'),
(7704,1124,'market_25'),
(2438,701,'market_min'),
(6496,700,'market_min'),
(6496,701,'market_min'),
(3670,700,'market_25'),
(3670,701,'market_25'),
(3670,704,'market_min'),
(2328,704,'market_min'),
(8649,1126,'market_min'),
(6346,700,'market_50'),
(7564,700,'market_min'),
(8906,700,19.4170467179223),
(7191,1125,'market_min'),
(1765,700,'market_min'),
(9711,700,'market_min'),
(10347,700,39.3829147724996),
(8721,701,19.1638743455497),
(7624,700,'market_min'),
(7624,701,'market_min'),
(8904,700,30.5235564755707),
(7521,700,'market_25'),
(6570,701,'market_min'),
(7524,700,'market_25'),
(10596,700,'market_min'),
(10596,703,'market_min'),
(10594,700,'market_min'),
(6965,701,'market_min'),
(6965,703,'market_min'),
(2726,1123,'market_min'),
(12234,701,'market_min'),
(10940,1125,'market_min'),
(96,700,'market_min'),
(96,1126,'market_min'),
(6268,1123,'market_min'),
(5733,1126,'market_50'),
(354,700,'market_50'),
(8945,703,'market_min'),
(8945,704,'market_min'),
(3982,703,'market_min'),
(974,702,'market_25'),
(13260,704,'market_25'),
(1813,701,'market_min'),
(2217,704,'market_min'),
(4001,704,'market_min'),
(13323,704,'market_50'),
(13323,1125,'market_50'),
(4955,700,'market_min'),
(12497,704,'market_75'),
(4175,701,'market_25'),
(4175,703,'market_min'),
(22330,701,'market_min'),
(23238,701,'market_min'),
(5722,700,'market_min'),
(21091,1124,'market_min'),
(23248,702,'market_min'),
(23248,704,'market_min'),
(23239,701,'market_min'),
(13326,704,'market_50'),
(21187,700,'market_min'),
(6936,1123,'market_min'),
(19685,701,'market_75'),
(19685,703,'market_min'),
(19685,704,'market_50'),
(19685,1123,'market_min'),
(2774,1124,'market_min'),
(10938,701,'market_25'),
(19687,703,'market_min'),
(19687,704,'market_25'),
(4678,701,'market_25'),
(2734,701,'market_25'),
(4715,701,'market_min'),
(3073,703,'market_min'),
(7707,701,'market_min'),
(9783,701,'market_50'),
(9783,703,'market_25'),
(9829,700,'market_min'),
(9829,704,'market_min'),
(13488,704,'market_min'),
(9760,701,'market_min'),
(9760,704,'market_min'),
(5076,702,'market_min'),
(2992,700,'market_25'),
(2992,703,'market_min'),
(2992,704,'market_min'),
(2992,1123,'market_min'),
(2992,1125,'market_min'),
(2992,1126,'market_min'),
(10717,700,'market_min'),
(10717,701,'market_min'),
(1777,701,'market_25'),
(2105,700,'market_min'),
(10398,703,'market_min'),
(10398,1125,'market_25'),
(10537,700,'market_25'),
(10722,701,'market_50'),
(10722,703,'market_50'),
(10722,704,'market_50'),
(10722,1125,'market_50'),
(823,703,'market_min'),
(2528,703,'market_min'),
(8157,1125,'market_min'),
(10936,701,'market_50'),
(10936,1123,'market_min'),
(7004,700,'market_25'),
(7004,701,'market_25'),
(7004,703,'market_min'),
(7004,1123,'market_min'),
(1446,1123,'market_25'),
(418,703,'market_min'),
(10124,700,'market_25'),
(10124,701,'market_min'),
(10124,1123,'market_min'),
(7005,703,'market_min'),
(2216,700,'market_min'),
(2216,704,'market_min'),
(2216,1123,'market_min'),
(5097,700,204.265228223481),
(13025,703,'market_25'),
(10616,701,'market_25'),
(10616,702,'market_25'),
(8898,700,'market_min'),
(8898,701,'market_min'),
(8915,1123,'market_25'),
(996,701,'market_min'),
(996,703,'market_min'),
(997,703,'market_min'),
(307,703,'market_min'),
(8106,700,'market_min'),
(2835,703,'market_min'),
(2835,704,'market_min'),
(593,700,'market_min'),
(523,703,'market_min'),
(957,1123,'market_50'),
(144,701,'market_min'),
(563,700,'market_75'),
(4964,700,'market_min'),
(11510,703,'market_min'),
(11836,1123,'market_min'),
(11836,1124,'market_min'),
(11836,1125,'market_min'),
(9289,702,'market_min'),
(10721,704,'market_min'),
(3909,701,'market_min'),
(3909,703,'market_min'),
(1406,700,'market_25'),
(1406,701,'market_min'),
(4032,702,'market_min'),
(2934,704,'market_min'),
(344,702,'market_min'),
(3900,701,'market_25'),
(3900,704,'market_min'),
(11835,1123,'market_min'),
(1120,700,'market_min'),
(1413,704,'market_min'),
(3898,700,'market_25'),
(3898,701,'market_min'),
(3898,703,'market_min'),
(3898,704,'market_min'),
(10529,1125,'market_min'),
(1294,704,'market_25'),
(3392,701,'market_min'),
(8938,704,'market_min'),
(8938,1123,'market_min'),
(10524,700,'market_min'),
(10524,701,'market_min'),
(4043,702,'market_min'),
(4043,704,'market_min'),
(4043,1124,'market_min'),
(4043,1125,'market_min'),
(13266,701,'market_50'),
(8990,703,'market_min'),
(8990,1125,'market_min'),
(2778,701,'market_min'),
(2778,1125,'market_min'),
(8853,701,'market_min'),
(2497,1125,'market_min'),
(6935,1123,'market_min'),
(2436,700,'market_min'),
(2436,704,'market_min'),
(6395,700,'market_75'),
(6395,701,'market_75'),
(12656,701,'market_75'),
(12791,701,'market_min'),
(12791,1125,'market_min'),
(12791,1126,'market_min'),
(13853,700,'market_25'),
(11788,700,'market_min'),
(3575,700,'market_min'),
(12621,700,'market_min'),
(12621,703,'market_min'),
(12621,704,'market_min'),
(12621,1124,'market_min'),
(19223,700,617.801047120419),
(12993,700,'market_min'),
(12993,1123,'market_min'),
(12994,703,'market_min'),
(6398,701,'market_25'),
(6339,700,'market_25'),
(13041,703,'market_min'),
(5644,703,'market_min'),
(2735,701,'market_min'),
(4716,703,'market_min'),
(2730,703,'market_25'),
(3581,703,'market_min'),
(11339,703,'market_min'),
(4720,703,'market_25'),
(10382,703,'market_25'),
(3988,704,'market_min'),
(950,701,'market_min'),
(950,703,'market_min'),
(950,704,'market_min'),
(4203,701,'market_25'),
(12494,703,'market_50'),
(12494,704,'market_50'),
(12494,1126,'market_50'),
(1091,702,'market_min'),
(4223,703,'market_min'),
(10941,701,'market_min'),
(10941,703,'market_min'),
(10941,1125,'market_25'),
(11894,700,'market_25'),
(11689,703,'market_min'),
(8636,701,'market_min'),
(9820,703,'market_min'),
(9372,703,'market_min'),
(6158,703,'market_min'),
(3456,701,'market_min'),
(3456,703,'market_min'),
(8674,700,'market_min'),
(3559,1126,'market_50'),
(9877,701,'market_min'),
(186,1125,'market_min'),
(3477,1124,'market_25'),
(8907,700,'market_min'),
(8907,701,'market_min'),
(8907,1123,'market_min'),
(8943,701,'market_25'),
(10406,701,'market_min'),
(4940,702,'market_min'),
(1776,700,'market_25'),
(1776,701,'market_25'),
(3981,703,'market_min'),
(1523,701,'market_min'),
(1523,1125,'market_min'),
(9380,704,'market_min'),
(9380,1125,'market_25'),
(3976,703,'market_min'),
(3976,704,'market_min'),
(415,701,'market_25'),
(8156,700,'market_min'),
(8156,1125,'market_min'),
(6030,701,'market_min'),
(13293,700,'market_75'),
(3987,704,'market_min'),
(4176,700,'market_min'),
(4176,703,'market_50'),
(4959,701,'market_25'),
(3430,701,'market_min'),
(19879,1123,150.523089589737),
(60,1123,'market_min'),
(10527,700,'market_25'),
(10527,701,'market_25'),
(12979,702,'market_min'),
(9827,701,'market_50'),
(9827,703,'market_min'),
(13324,700,'market_50'),
(13324,702,'market_50'),
(13324,703,'market_50'),
(13324,704,'market_50'),
(11093,702,'market_min'),
(7120,704,'market_25'),
(3989,704,'market_min'),
(9204,1124,'market_25'),
(11886,1123,'market_min'),
(11886,1124,'market_min'),
(3722,1124,'market_25'),
(11402,701,'market_min'),
(11402,702,'market_min'),
(11402,1123,'market_min'),
(11402,1125,'market_min'),
(3258,703,'market_min'),
(11399,701,'market_min'),
(9358,700,'market_min'),
(9358,701,'market_25'),
(9358,1124,'market_min'),
(9358,1125,'market_min'),
(6931,701,'market_min'),
(11681,700,'market_25'),
(11681,701,'market_25'),
(11681,702,'market_25'),
(11681,703,'market_25'),
(11681,704,'market_25'),
(11681,1123,'market_25'),
(11681,1124,'market_25'),
(11681,1125,'market_25'),
(11681,1126,'market_50'),
(11683,1123,'market_50'),
(11683,1124,'market_25'),
(11685,700,'market_50'),
(11685,701,'market_50'),
(11685,702,'market_50'),
(11685,703,'market_25'),
(11685,704,'market_25'),
(11685,1123,'market_50'),
(11685,1124,'market_25'),
(11685,1125,'market_25'),
(11685,1126,'market_25'),
(11684,701,'market_50'),
(11684,702,'market_50'),
(11684,703,'market_min'),
(11684,704,'market_25'),
(11684,1123,'market_50'),
(11684,1125,'market_25'),
(11684,1126,'market_25'),
(8648,704,'market_min'),
(8648,1123,'market_min'),
(8650,701,'market_min'),
(8650,703,'market_min'),
(8650,704,'market_min'),
(5494,701,'market_min'),
(6219,700,'market_75'),
(8586,701,29.3164397905759),
(7565,700,'market_min'),
(9841,701,'market_min'),
(9841,702,'market_min'),
(9841,1123,'market_min'),
(9841,1124,'market_min'),
(9840,703,'market_min'),
(9840,704,'market_min'),
(10587,701,'market_50'),
(10587,703,'market_50'),
(10587,704,'market_50'),
(10587,1123,'market_25'),
(8713,701,26.5938896559074),
(10484,700,'market_50'),
(6566,700,'market_min'),
(6566,701,'market_50'),
(6567,700,'market_25'),
(6567,701,'market_min'),
(6340,700,'market_75'),
(1448,701,763.432175346068),
(23032,701,'market_min'),
(23032,1123,'market_min'),
(23032,1124,'market_25'),
(5831,701,'market_25'),
(5831,703,'market_min'),
(10595,701,'market_min'),
(10595,703,'market_min'),
(3071,700,'market_min'),
(3079,701,'market_min'),
(3583,701,'market_25'),
(3047,700,'market_25'),
(3053,703,'market_min'),
(4076,700,'market_50'),
(8217,701,'market_min'),
(1052,1123,'market_min'),
(3984,703,'market_min'),
(19992,701,'market_min'),
(19996,701,'market_min'),
(19965,704,'market_min'),
(4956,701,'market_min'),
(10411,704,'market_min'),
(3979,703,'market_min'),
(4957,700,'market_min'),
(6073,701,'market_75'),
(3906,701,'market_min'),
(4963,701,'market_min'),
(3899,700,'market_min'),
(4958,700,'market_25'),
(10580,704,'market_min'),
(10581,704,'market_min'),
(10581,1126,'market_min'),
(4965,700,'market_50'),
(4962,700,'market_min'),
(10530,700,'market_50'),
(10530,701,'market_min'),
(10530,1125,'market_min'),
(3,702,'market_min'),
(3,704,'market_50'),
(25,701,'market_25'),
(146,701,'market_50'),
(143,701,'market_50'),
(380,700,'market_50'),
(380,701,'market_50'),
(380,703,'market_50'),
(380,704,'market_50'),
(76,704,'market_min'),
(221,701,'market_min'),
(332,703,'market_min'),
(332,1124,'market_25'),
(334,700,'market_min'),
(409,700,'market_min'),
(410,700,'market_min'),
(439,703,'market_min'),
(412,701,'market_min'),
(412,703,'market_min'),
(524,701,'market_min'),
(738,703,'market_min'),
(738,1124,'market_min'),
(944,704,'market_min'),
(951,701,'market_min'),
(951,703,'market_min'),
(1560,701,'market_min'),
(1699,703,'market_min'),
(1701,702,'market_25'),
(1706,700,'market_25'),
(1706,701,'market_min'),
(1722,704,'market_50'),
(1722,1125,'market_50'),
(2054,701,'market_min'),
(1880,700,'market_75'),
(2668,703,'market_min'),
(2854,701,'market_min'),
(2879,704,'market_min'),
(3391,701,'market_min'),
(3391,704,'market_min'),
(3983,703,'market_min'),
(5153,703,'market_25'),
(10408,701,'market_min'),
(10408,703,'market_min'),
(11092,702,'market_min'),
(12212,701,'market_75'),
(12492,700,'market_50'),
(12492,703,'market_50'),
(12492,704,'market_50'),
(12493,703,'market_50'),
(12493,704,'market_50'),
(19987,700,'market_min'),
(11814,703,'market_50'),
(2213,704,'market_min'),
(2718,701,269.654929781965),
(2188,701,'market_min'),
(2188,702,'market_min'),
(2188,703,'market_min'),
(14041,701,'market_25'),
(14041,703,'market_min'),
(14041,1124,'market_min'),
(14041,1125,'market_min'),
(2775,704,'market_min'),
(19981,703,'market_25'),
(6218,700,'market_50'),
(6223,700,'market_50'),
(6223,701,'market_75'),
(10588,703,'market_50'),
(10588,704,'market_50'),
(10588,1123,'market_25'),
(4372,700,'market_min'),
(4372,701,'market_min'),
(10942,703,'market_min'),
(3889,701,'market_min'),
(3889,703,'market_min'),
(3892,702,'market_min'),
(3893,701,'market_25'),
(3893,703,'market_min'),
(3894,702,'market_min'),
(362,701,'market_min'),
(5946,700,'market_min'),
(1447,701,'market_min'),
(10646,700,35.2182595658594),
(10646,701,35.2182595658594),
(10626,700,'market_25'),
(10626,701,'market_75'),
(10487,701,'market_75'),
(10488,700,'market_50'),
(10488,701,'market_25'),
(12992,703,'market_min'),
(12992,1126,'market_min'),
(6778,701,'market_25'),
(7522,700,'market_25'),
(12646,704,'market_25'),
(11610,703,'market_min'),
(11059,700,'market_min'),
(11059,701,'market_min'),
(20208,701,'market_min'),
(20208,703,'market_min'),
(20220,701,'market_25'),
(20220,703,'market_min'),
(8677,701,'market_min'),
(8677,704,'market_min'),
(8283,1123,'market_min'),
(20650,701,'market_min'),
(20650,703,'market_min'),
(20650,704,'market_min'),
(3891,703,'market_min'),
(3891,704,'market_min'),
(2836,701,'market_min'),
(2836,1123,'market_min'),
(20963,703,'market_min'),
(1243,701,'market_min'),
(378,703,'market_min'),
(378,1125,'market_min'),
(378,1126,'market_min'),
(3434,701,'market_min'),
(3434,702,'market_25'),
(3434,1124,'market_min'),
(3434,1125,'market_min'),
(3434,1126,'market_min'),
(8651,700,'market_min'),
(8651,703,'market_min'),
(21081,704,'market_min'),
(19985,700,'market_min'),
(19985,701,'market_min'),
(7573,700,'market_50'),
(21706,702,28.5456396553402),
(21686,701,'market_min'),
(20964,700,'market_75'),
(22037,704,'market_min'),
(21773,704,365.899934839396),
(22317,701,'market_min'),
(22089,701,'market_min'),
(21319,704,'market_min'),
(22125,703,490.401308938416),
(21798,1123,'market_50'),
(22316,701,'market_min'),
(22316,703,'market_min'),
(141,701,'market_min'),
(1602,701,'market_50'),
(1602,704,'market_min'),
(22708,700,'market_min'),
(22708,701,'market_25'),
(22874,700,'market_25'),
(22871,701,'market_min'),
(22871,704,'market_min'),
(23276,700,'market_min'),
(23276,701,'market_25'),
(23275,701,'market_50'),
(20969,700,'market_min'),
(21710,703,54.5125226831902),
(21710,704,54.5125226831902),
(2776,704,'market_min'),
(2776,1123,'market_min'),
(2776,1125,'market_min'),
(1504,1123,'market_min'),
(23994,702,'market_min'),
(23992,1125,'market_min'),
(22362,700,'market_min'),
(22362,701,'market_min'),
(22325,700,'market_50'),
(22325,1125,'market_min'),
(2190,700,'market_25'),
(2190,701,'market_min'),
(2190,702,'market_min'),
(2190,704,'market_min'),
(2190,1123,'market_min'),
(23602,700,'market_min'),
(23602,701,'market_min'),
(24228,704,'market_min'),
(229,703,'market_min'),
(229,704,'market_min'),
(24664,701,'market_25'),
(24664,702,'market_25'),
(13264,701,'market_25'),
(13264,704,'market_min'),
(24670,703,'market_min'),
(24670,1123,'market_min'),
(24669,700,'market_25'),
(24669,704,'market_min'),
(24127,701,'market_min'),
(22361,1124,'market_min'),
(24844,701,'market_75'),
(24707,701,'market_min'),
(21077,700,'market_min'),
(21077,701,'market_min'),
(21946,700,'market_min'),
(24245,700,'market_50'),
(24245,1126,'market_50'),
(25278,700,'market_75'),
(25278,701,'market_50'),
(24142,700,565.444801010178),
(23336,701,'market_25'),
(2733,703,'market_25'),
(2736,700,'market_min'),
(2736,701,'market_min'),
(24125,701,'market_min'),
(2335,701,'market_50'),
(2335,703,'market_25'),
(25718,704,'market_50'),
(25698,1123,35.1961438268666),
(25698,1126,35.1961438268666),
(25695,1124,162.127837758025),
(25694,1125,162.130383999305),
(25697,1124,162.132348720958),
(25713,703,'market_50'),
(25713,704,'market_50'),
(25713,1125,'market_50'),
(25712,703,'market_50'),
(25712,704,'market_50'),
(25711,1125,'market_50'),
(25709,703,'market_50'),
(25709,704,'market_75'),
(25710,700,'market_50'),
(25710,704,'market_50'),
(25710,1125,'market_50'),
(25899,701,'market_50'),
(25765,700,'market_min'),
(25958,700,'market_25'),
(25960,700,'market_25'),
(25991,700,'market_25'),
(8286,701,'market_min'),
(8286,703,'market_min'),
(22327,701,'market_min'),
(22327,703,'market_min'),
(22327,1124,'market_min'),
(22327,1126,'market_min'),
(14039,701,'market_min'),
(14039,703,'market_25'),
(14039,1123,'market_min'),
(25994,700,'market_min'),
(26126,703,'market_25'),
(26126,704,'market_min'),
(26126,1123,'market_min'),
(26126,1125,'market_min'),
(26127,702,'market_min'),
(26127,703,'market_min'),
(26127,704,'market_min'),
(26127,1125,'market_min'),
(26127,1126,'market_min'),
(26125,703,'market_min'),
(26125,704,'market_min'),
(26125,1125,'market_min'),
(26392,700,'market_min'),
(26392,703,'market_min'),
(26392,704,'market_min'),
(25345,701,'market_min'),
(27219,703,'market_min'),
(27214,701,'market_min'),
(27213,701,'market_min'),
(27212,701,'market_min'),
(6224,700,'market_50'),
(6209,700,98.6181135031452),
(6207,700,'market_min'),
(6207,701,'market_50'),
(6222,700,'market_min'),
(25437,700,'market_50'),
(936,1125,'market_min'),
(25634,700,'market_25'),

]


# =============================================================================
# COMPUTE NEW PRICES (per product × cohort)
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'market_min',
    'market_25': 'market_25',
    'market_50': 'market_50',
    'market_75': 'market_75',
    'market_max': 'market_max',
    'market_avg': 'market_avg',
}

results = []
for product_id,cohort_id, action in PUSH_LIST:
    product_rows = lookup[(lookup['product_id'] == product_id)&(lookup['cohort_id'] == cohort_id)] #
    if product_rows.empty:
        results.append({'product_id': product_id, 'cohort_id': '-', 'action': str(action), 'status': 'NOT FOUND'})
        continue

    for _, row in product_rows.iterrows():
        cohort_id = int(row['cohort_id'])
        wac = row.get('wac_p', None)
        sku = row.get('sku', '')
        brand = row.get('brand', '')

        if isinstance(action, (int, float)):
            base_price = float(action)
            action_label = f'fixed_{action}'
        elif action == 'target_margin':
            tm = row.get('target_margin', 0.05)
            if pd.isna(wac) or wac <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
                continue
            base_price = wac / (1 - tm)
            action_label = f'target_margin ({tm:.1%})'
        elif action == 'step_up':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_up = [t for t in tiers if t > cur + 0.25]
            if not next_up:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT TOP'})
                continue
            base_price = next_up[0]
            action_label = f'step_up ({cur:.2f} -> {base_price:.2f})'
        elif action == 'step_down':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_down = [t for t in reversed(tiers) if t < cur - 0.5]
            if not next_down:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT BOTTOM'})
                continue
            base_price = next_down[0]
            action_label = f'step_down ({cur:.2f} -> {base_price:.2f})'
        elif action in ACTION_TO_COLUMN:
            col = ACTION_TO_COLUMN[action]
            val = row.get(col, None)
            if pd.isna(val):
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
                continue
            base_price = val
            action_label = action
        else:
            results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
            continue

        base_price_rounded = np.round(base_price * 4) / 4
        margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

        cur_price = row.get('current_price', None)

        results.append({
            'product_id': product_id,
            'cohort_id': cohort_id,
            'sku': sku,
            'brand': brand,
            'action': action_label,
            'wac': wac,
            'current_price': cur_price,
            'base_price': base_price,
            'new_price': base_price_rounded,
            'margin': margin,
            'status': 'OK',
        })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
products_ok = df_results[df_results['status'] == 'OK']['product_id'].nunique()
print(f'Results: {ok_count} OK across {products_ok} products × cohorts, {err_count} errors/skipped')
if err_count > 0:
    print('\nErrors/Skipped:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'cohort_id', 'sku', 'action', 'status']].to_string(index=False))
df_results[df_results['status'] == 'OK']

Results: 761 OK across 488 products × cohorts, 0 errors/skipped


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status
0,8285,703,صابون كامى كلاسيك -- 110 جم,كامى,market_min,63.242901,64.00,63.250,63.25,0.000112,OK
1,21269,701,شورتى بسكويت - 10 جنية,بسكاتو,market_min,88.826989,92.25,89.000,89.00,0.001944,OK
2,21269,703,شورتى بسكويت - 10 جنية,بسكاتو,market_min,88.826989,98.75,96.500,96.50,0.079513,OK
3,12675,701,توشيبا ازرق طورش 2حجر شرنك D 2 حجر,توشيبا,market_50,259.729386,294.75,285.125,285.00,0.088669,OK
4,2287,1124,حفاضات بامبرز عبوة التوفير مقاس 2 - 60 حفاضة,بامبرز,market_min,276.814195,294.75,294.500,294.50,0.060054,OK
...,...,...,...,...,...,...,...,...,...,...,...
756,6207,701,برجر حلوانى بيف 20 قطعة - 1 كجم,حلوانى اخوان,market_50,212.981119,252.25,242.250,242.25,0.120821,OK
757,6222,700,دجاج كوكى كرانشى بارد + بطاطس - 12 قطعه,كوكى,market_min,192.123999,212.00,202.250,202.25,0.050067,OK
758,25437,700,سفن اب اكسترا فيز بلاستيك - 400 مل,بيبسي,market_50,103.494864,110.00,106.500,106.50,0.028217,OK
759,936,1125,اولكر بسكويت شاي- 5 ج,اولكر,market_min,45.448559,48.75,48.500,48.50,0.062916,OK


In [6]:
df_results=df_results[df_results['current_price']>df_results['new_price']]
df_results['change'] = (df_results['new_price'] - df_results['current_price'])/df_results['current_price']
print(df_results['margin'].mean())
print(df_results['margin'].median())
print(df_results['margin'].min())
print(df_results['margin'].max())
#df_results=df_results[~df_results['brand'].isin(['كوكا كولا','شويبس'])]
df_results

0.06521332850173692
0.05996681987761318
-0.0633077577741052
0.2860474696134896


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status,change
0,8285,703,صابون كامى كلاسيك -- 110 جم,كامى,market_min,63.242901,64.00,63.250,63.25,0.000112,OK,-0.011719
1,21269,701,شورتى بسكويت - 10 جنية,بسكاتو,market_min,88.826989,92.25,89.000,89.00,0.001944,OK,-0.035230
2,21269,703,شورتى بسكويت - 10 جنية,بسكاتو,market_min,88.826989,98.75,96.500,96.50,0.079513,OK,-0.022785
3,12675,701,توشيبا ازرق طورش 2حجر شرنك D 2 حجر,توشيبا,market_50,259.729386,294.75,285.125,285.00,0.088669,OK,-0.033079
4,2287,1124,حفاضات بامبرز عبوة التوفير مقاس 2 - 60 حفاضة,بامبرز,market_min,276.814195,294.75,294.500,294.50,0.060054,OK,-0.000848
...,...,...,...,...,...,...,...,...,...,...,...,...
756,6207,701,برجر حلوانى بيف 20 قطعة - 1 كجم,حلوانى اخوان,market_50,212.981119,252.25,242.250,242.25,0.120821,OK,-0.039643
757,6222,700,دجاج كوكى كرانشى بارد + بطاطس - 12 قطعه,كوكى,market_min,192.123999,212.00,202.250,202.25,0.050067,OK,-0.045991
758,25437,700,سفن اب اكسترا فيز بلاستيك - 400 مل,بيبسي,market_50,103.494864,110.00,106.500,106.50,0.028217,OK,-0.031818
759,936,1125,اولكر بسكويت شاي- 5 ج,اولكر,market_min,45.448559,48.75,48.500,48.50,0.062916,OK,-0.005128


In [10]:
mona_query = '''
with base as (
select  dt.taggable_id as retailer_id,
		dynamic_tags.name as dynamic_tag_name, 
		dynamic_tags.id as dynamic_tag_id, 
		c.id as new_cohort_id, 
		c.name as new_cohort_name
from dynamic_tags
inner join dynamic_taggables dt on dt.dynamic_tag_id = dynamic_tags.id
left join cohorts c on c.dynamic_tag_id = dynamic_tags.id
where dynamic_tags.name in ('mona_700', 'mona_701', 'mona_702', 'mona_703', 'mona_704', 'mona_1123', 'mona_1124', 'mona_1125', 'mona_1126', 'mona_cairo')), 

retailer_cohorts as (
select * 
from (
select distinct	
	taggable_id as retailer_id ,
	case when custom_ct =1 then cohort_id end as custom_cohort,
	case when custom_ct = 1 then fallbk else cohort_id end as main_cohort,
	61 as general_cohort, 
	dynamic_tag_name, 
	new_cohort_id, 
	new_cohort_name
from (
select 
	taggable_id,
	cohorts.id as cohort_id,
	FALLBACK_COHORT_ID as fallbk,
	cohorts.priority,
	case when coalesce(FALLBACK_COHORT_ID,61) = 61 then 0 else 1 end as custom_ct,
	rank () over (partition by taggable_id,custom_ct order by priority) as rk,
	sum (custom_ct) over (partition by taggable_id) as custom_count, 
	dynamic_tag_name, 
	new_cohort_id, 
	new_cohort_name
from cohorts
	left join dynamic_taggables as dt on dt.dynamic_tag_id  = cohorts.dynamic_tag_id
	inner join base on base.retailer_id = dt.taggable_id 
where cohorts.is_active = true and cohorts.id <> 61 
) as sub 
where rk = 1 and (custom_count = 0 or (custom_count >0 and custom_ct = 1))
order by 1)), 

served_warehouses as (
select  warehouse_id, 
		new_cohort_id, 
		count(distinct parent_sales_order_id) order_counts
from sales_orders 
inner join retailer_cohorts on retailer_cohorts.retailer_id = sales_orders.retailer_id 
where sales_orders.created_at::date between current_date - 90 and current_date 
group by all ), 

historical_stocks as (
select  distinct product_id, new_cohort_id
from materialized_views.stock_day_close std
inner join served_warehouses sw on sw.warehouse_id = std.warehouse_id 
where available_stock > 1
and timestamp::date between current_date - 30 and current_date), 

stock_now as (
select distinct product_id, new_cohort_id
from product_warehouse 
inner join served_warehouses sw on sw.warehouse_id = product_warehouse.warehouse_id 
where available_stock > 0), 

cohort_fallbacks as (
select  distinct custom_cohort, 
		main_cohort, 
		general_cohort, 
		new_cohort_id, 
		new_cohort_name
from retailer_cohorts), 

final as (
select  product_id as "Product ID", 
		product_name as "Product Name", 
		packing_unit_id as "Packing Unit ID", 
		price as "Price", 
		case when visibility = 1 then 'YES' else 'NO' end as "Visibility (YES/NO)", 
		null as "Execute At (format:dd/mm/yyyy HH:mm)", 
		null as "Tags", 
		new_cohort_id, 
		new_cohort_name
from (
select s.*,
	   coalesce(case when cppu3.is_customized = true then  cppu3.price end, cppu2.price,cppu.price) as price, 
	   case when c.section_id in (714,626, 516, 417, 351,121,285,20,54,37,10,43,44,36,14,8,55,619,622,621) and (stock_now.product_id is not null or historical_stocks.product_id is not null) then 1 else 0 end as visibility, 
	   p.name_ar as product_name
from (
select product_id, 
       packing_unit_id, 
	   custom_cohort, 
	   main_cohort, 
	   general_cohort, 
	   new_cohort_id, 
	   new_cohort_name, 
	   packing_unit_products.id as product_packing_unit_id
from packing_unit_products
cross join cohort_fallbacks
where DELETEd_AT is null
) s 
left join products p on p.id = s.product_id 
left join categories c on c.id = p.category_id 
left join cohort_product_packing_units as cppu on cppu.cohort_id = s.general_cohort and cppu.product_packing_unit_id = s.product_packing_unit_id
left join cohort_product_packing_units as cppu2 on cppu2.cohort_id = s.main_cohort and cppu2.product_packing_unit_id = s.product_packing_unit_id
left join cohort_product_packing_units as cppu3 on cppu3.cohort_id = s.custom_cohort and cppu3.product_packing_unit_id = s.product_packing_unit_id
left join historical_stocks on historical_stocks.product_id = s.product_id and historical_stocks.new_cohort_id = s.new_cohort_id
left join stock_now on stock_now.product_id = s.product_id and stock_now.new_cohort_id = s.new_cohort_id
)) 


select final.*,
       cppu.price AS db_price
from final
join packing_unit_products pup
  on pup.product_id = final."Product ID"
 and pup.packing_unit_id = final."Packing Unit ID"
left join cohort_product_packing_units as cppu
  on cppu.cohort_id = final.new_cohort_id
 and cppu.product_packing_unit_id = pup.id
where cppu.price <> final."Price"
'''
mona_data = query_snowflake(mona_query)
len(mona_data)

0

In [ ]:
for cohort_id in mona_data.NEW_COHORT_ID.unique():
    out = mona_data[mona_data['NEW_COHORT_ID'] == cohort_id]
    id_ = cohort_id
    name = out.NEW_COHORT_NAME.values[0]
    file_name_ = 'uploads/1_new_{}.xlsx'.format(name).replace(' ','_')
    out.to_excel(file_name_,index = False)
    ################### Loop to avoid file limit ######################
    # split file into chunks
    print('Spliting file into chunks...')
    if id_ == 61:
        chunks = [out[i:i + 2000] for i in range(0, len(out), 2000)]
    else:
        chunks = [out[i:i + 4000] for i in range(0, len(out), 4000)]
    print(f'len chunks = {len(chunks)}')
    fileslist = []
    for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
        fileslist.append(f'manual/output_{id_}_chunk_{i + 1}.xlsx')
        output_file_path = f'manual/output_{id_}_chunk_{i + 1}.xlsx'
        chunk.to_excel(output_file_path, index=False, engine='xlsxwriter')
    # Loop over chunks and upload
    print('Uploading...')
    for file in fileslist:
        chunk = file.split('chunk_')[1].split('.xls')[0]
        x = post_prices(id_, file)
        # print(str(x.content))
        if ('"success":true' in str(x.content).lower()):
            print(f"Prices are upoladed successfuly Region: {name} ,cohort: {id_}, chunk: {chunk}")
        else:
            print(f"ERROR Region: {name} ,cohort: {id_}, chunk: {chunk}")
            print(x.content)
            final_status = False
            break

In [7]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()

# Filter to only SKUs with stock
if len(df_ok) > 0:
    stock_by_product = df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    df_ok = df_ok.merge(stock_by_product, on='product_id', how='left')
    no_stock = df_ok['stocks'].fillna(0) <= 0
    if no_stock.any():
        print(f'Skipping {no_stock.sum()} rows with no stock')
        df_ok = df_ok[~no_stock].copy()

if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        target_cohort = int(r['cohort_id'])
        cohort_whs = wh_df[wh_df['cohort_id'] == target_cohort]

        if cohort_whs.empty:
            cohort_whs = pd.DataFrame([{'warehouse_id': 1, 'cohort_id': target_cohort}])

        cur_price = r['current_price'] if pd.notna(r.get('current_price')) else 0

        for _, wh in cohort_whs.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': target_cohort,
                'stocks': 1,
                'current_price': cur_price,
            })

    df_push = pd.DataFrame(push_rows)

    n_products = df_ok['product_id'].nunique()
    n_cohorts = df_ok['cohort_id'].nunique()
    print(f'Pushing {n_products} products × {n_cohorts} cohorts = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')

Pushing 488 products × 9 cohorts = 1198 rows
Mode: live


🚀 MODE: LIVE
   Files will be prepared AND uploaded to API
Loading disable_pu_visibility from Google Sheets...
  ✓ Loaded 88 products to disable min PU visibility

PUSH PRICES - Source: manual_push
Total received: 1198
Price changes to push: 1198
Skipped (no change): 0

Price changes summary:
  Increases: 0
  Decreases: 1198

🔗 Mirrored prices to 6 main/general cohorts (+960 rows)
   Cohort 695 ← 700: 230 rows
   Cohort 61 ← 700: 230 rows
   Cohort 699 ← 702: 57 rows
   Cohort 697 ← 703: 211 rows
   Cohort 698 ← 704: 164 rows
   Cohort 696 ← 1123: 68 rows

📋 Prepared 2170 packing unit prices (incl. main cohorts)

Processing cohort: 61
  Saved: uploads/manual_push_61.xlsx (230 rows)
  Split into 1 chunks (size: 2000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 54.85it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 695
  Saved: uploads/manual_push_695.xlsx (230 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 55.21it/s]

...

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 1126
  Saved: uploads/manual_push_1126.xlsx (37 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 153.51it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 2170
Total failed: 0
  Cleanup: removed 124 temporary .xlsx files from 2 folder(s)

Done. Result: {'total_received': 1198, 'price_changes': 1198, 'pushed': 2170, 'failed': 0, 'skipped': 0, 'source_module': 'manual_push', 'timestamp': '2026-04-22 13:00:41', 'mode': 'live', 'skipped_cohorts': []}
